In [1]:
import pandas as pd

from metrics.process_data import metrics_processor

/home/javi/e2r_metrics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to /home/javi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/javi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_excel("comparation.xlsx", sheet_name="Matriz comparativa")

In [3]:
df.iloc[28]

#                                                                                                             NaN
Sección                                                         TEXTO ADAPTADO (SALIDA DE CADA HERRAMIENTA DE LF)
Original                                                        AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO\n\nD....
Lectura Fácil [Referencia]                                      Permiso para usar mis fotos o vídeos\nDatos pe...
FACILE                                                          Autorización de difusión de imagen/ vídeo.\n"D...
Placea                                                          AUTORIZACIÓN PARA USAR IMÁGENES O VÍDEOS\n\nYo...
Asistente de lectura fácil “ANTONIO GONZALES CRESPO”            AUTORIZACIÓN DE USO DE IMAGEN Y VÍDEO\nDatos d...
Asistente de lectura fácil “Mark Jonathan Camacho Escatel”:     AUTORIZACIÓN PARA USAR IMÁGENES Y VÍDEOS\n\nYo...
AdaptaTuTexto (Gemini 3.1 Pro)\nMuy A2                          Permiso para usar mis fo

In [4]:
df.iloc[28]["Original"]

'AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO\n\nD./Dña. \t\t\t\tcon DNI / Pasaporte nº        \t  y con email (a efectos de tratamiento de los datos): \t                                   , autoriza a la Universidad Politécnica de Madrid, CIF Q-2818015-F (en adelante, UPM), \n a la difusión de las imágenes o de la grabación de su autoría/participación con título: \nCON INCLUSENS TODAS LAS OPINIONES CUENTAN\n\nen el Proyecto Aprendizaje-Servicio__ Desarrollo de herramientas inclusivas para personas con discapacidad cognitiva en las pruebas sensoriales de consumidores (INCLUSENS) 2.0___.\nSe prevé la posible difusión de este material audiovisual, incluido Internet, con sujeción a la finalidad del tratamiento descrita, que excluye su uso comercial. Los canales utilizados serán: el canal oficial de Youtube de la UPM, el portal de ApS-UPM (https://aprendizajeservicio.upm.es/), de innovación educativa (https://innovacioneducativa.upm.es) y en la red social de X (@IEducativa_UPM) y Linkedin (www

In [5]:
new_df = df.iloc[28].to_frame(name="prediction_text")
new_df.rename(columns={"28":"prediction_text"}, inplace=True)
new_df["original_text"] = df.iloc[28]["Original"]
new_df["simplified_text"] = df.iloc[28]["Lectura Fácil [Referencia]"]
new_df["document_id"] = new_df.index
new_df["original_sentence_id"] = new_df.index

In [6]:
new_df.columns

Index(['prediction_text', 'original_text', 'simplified_text', 'document_id',
       'original_sentence_id'],
      dtype='str')

In [7]:
df_to_calculate_metric = new_df[4:]

In [8]:
import re


def clean_text(text):
    if not isinstance(text, str) or pd.isna(text):
        return ""
    
    # --- A. CARACTERES INVISIBLES Y VIÑETAS EXTRAÑAS ---
    text = text.replace('\xa0', ' ').replace('\t', ' ')
    text = text.replace('\n', ' ').replace('\r', ' ')
    # Eliminar viñetas raras (símbolos unicode de Word/PDF)
    text = re.sub(r'[\uf02d\uf0a7\uf0b7•]', ' ', text)
    # Reemplazar comillas tipográficas y guiones especiales
    text = text.replace('“', '"').replace('”', '"').replace('‑', '-')
    
    # --- B. RUIDO DE FORMATO ---
    text = re.sub(r'_{2,}', ' ', text)
    text = re.sub(r'\.{3,}', '.', text)
    text = re.sub(r'-{2,}', ' ', text)
    text = text.replace('*', '')
    
    # --- C. COMILLAS Y NORMALIZACIÓN ---
    text = text.strip().strip('"').strip("'")
    text = text.replace('"', '')
    
    # --- D. PUNTUACIÓN INTELIGENTE ---
    # 1. Quitar espacios antes de puntuación
    text = re.sub(r'\s+([,.?;:])', r'\1', text)
    
    # 2. Añadir espacio tras comas, puntos y comas y dos puntos
    # Exclusión: no añadir si le sigue un número, espacio o slash (evita romper https://)
    text = re.sub(r'([,?;:])(?=[^\s\d/])', r'\1 ', text)
    
    # 3. Añadir espacio tras punto SOLO si le sigue mayúscula o corchete (evita D./Dña., 2.0)
    text = re.sub(r'\.(?=[A-ZÁÉÍÓÚ\[])', r'. ', text)
    
    # 4. Reparar URLs o correos dañados en procesamientos previos (si los hubiera)
    text = text.replace('https: //', 'https://').replace('http: //', 'http://')
    text = text.replace('. es', '.es').replace('. com', '.com').replace('. html', '.html')
    
    # --- E. ESPACIOS FINALES ---
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 2. Aplicar limpieza
for col in ['prediction_text', 'original_text', 'simplified_text']:
    if col in df_to_calculate_metric.columns:
        df_to_calculate_metric[col] = df_to_calculate_metric[col].apply(clean_text)

# 3. Filtro de longitud y exportación
df_to_calculate_metric = df_to_calculate_metric[df_to_calculate_metric['prediction_text'].str.len() > 5]

# Guardamos el archivo limpio para que MetricsProcessor lo use sin errores
df_to_calculate_metric.to_excel('clean.xlsx', index=False)

print(f"Limpieza finalizada. Filas listas para procesar: {len(df_to_calculate_metric)}")
df_to_calculate_metric.head()


Limpieza finalizada. Filas listas para procesar: 16


,prediction_text,original_text,simplified_text,document_id,original_sentence_id
FACILE,Autorización de difusión de imagen/ vídeo. D./...,AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO D./Dñ...,Permiso para usar mis fotos o vídeos Datos per...,FACILE,FACILE
Placea,"AUTORIZACIÓN PARA USAR IMÁGENES O VÍDEOS Yo, D...",AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO D./Dñ...,Permiso para usar mis fotos o vídeos Datos per...,Placea,Placea
Asistente de lectura fácil “ANTONIO GONZALES CRESPO”,AUTORIZACIÓN DE USO DE IMAGEN Y VÍDEO Datos de...,AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO D./Dñ...,Permiso para usar mis fotos o vídeos Datos per...,Asistente de lectura fácil “ANTONIO GONZALES C...,Asistente de lectura fácil “ANTONIO GONZALES C...
Asistente de lectura fácil “Mark Jonathan Camacho Escatel”:,"AUTORIZACIÓN PARA USAR IMÁGENES Y VÍDEOS Yo, N...",AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO D./Dñ...,Permiso para usar mis fotos o vídeos Datos per...,Asistente de lectura fácil “Mark Jonathan Cama...,Asistente de lectura fácil “Mark Jonathan Cama...
AdaptaTuTexto (Gemini 3.1 Pro)\nMuy A2,Permiso para usar mis fotos y vídeos Mis datos...,AUTORIZACIÓN DE DIFUSIÓN DE IMAGEN/VÍDEO D./Dñ...,Permiso para usar mis fotos o vídeos Datos per...,AdaptaTuTexto (Gemini 3.1 Pro)\nMuy A2,AdaptaTuTexto (Gemini 3.1 Pro)\nMuy A2


In [9]:
metric = metrics_processor(df_to_calculate_metric)
metric.save_consolidated_report("metric.xlsx")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 737.75it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 492.33it/s]
BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm

Reporte consolidado guardado en: metric.xlsx
